In [ ]:
import os
import sys
import math
import torch
import random
import numpy as np
import torch.nn as nn
from pathlib import Path
import torch.nn.functional as F
import matplotlib.pyplot as plt
from einops import rearrange, reduce, repeat, einsum


import training_engine  
import inference_engine
from utils import RMSNorm, init_custom_weight



MODEL_CONFIGS = {
    "embed_dim"   : 2**5,   # 32
    "max_seq_len" : 2**6,   # 64
    "vocab_size"  : 2**11,  # 2048
    "num_attn_heads": 1,
    "num_layers": 3,
   
    "batch_size"  : 2**4,   # 16
    "learning_rate" : 1e-3,

    "save_path" : "../models/level3_single_head_attn.pt",
    "loss_history" : "../models/level3_single_head_attn_loss_history.txt",

}



# Device configuration
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)




cuda


In [ ]:
class Level3SingleHeadCausalAttention_Model(nn.Module):

    # embed, PE, blocks, final norm, unembed

    def __init__(self, config=MODEL_CONFIGS):
        super().__init__()
        ...


    def forward():
        ...

In [ ]:
class Level3Transformer_Block(nn.Module):

    # RMS norm, attention, residual

    def __init__(self, config):
        super().__init__()
        ...


    def forward():
        ...


In [ ]:
class SingleHeadCausalAttention_SubBlock(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.embed_dim = config["embed_dim"]
        self.num_attn_heads = config["num_attn_heads"]
        self.head_dim = self.embed_dim // self.num_attn_heads
        self.num_layers = config["num_layers"]          # needed for residual scaling

        assert self.embed_dim % self.num_attn_heads == 0, \
            "embed_dim must be divisible by num_attn_heads"

        # Weight matrices stored as (in_features, out_features)
        #   query/key/value : (embed_dim, num_heads * head_dim)
        #   out             : (num_heads * head_dim, embed_dim)
        self.query_w = nn.Parameter(torch.empty(self.embed_dim, self.num_attn_heads * self.head_dim))
        self.key_w   = nn.Parameter(torch.empty(self.embed_dim, self.num_attn_heads * self.head_dim))
        self.value_w = nn.Parameter(torch.empty(self.embed_dim, self.num_attn_heads * self.head_dim))
        self.out_w   = nn.Parameter(torch.empty(self.num_attn_heads * self.head_dim, self.embed_dim))

        self.reset_parameters()

        # Causal mask (True = positions that must be masked)
        mask = torch.triu(
            torch.ones(config["max_seq_len"], config["max_seq_len"], dtype=torch.bool),
            diagonal=1,
        )
        self.register_buffer("mask", mask, persistent=False)

    def reset_parameters(self):
        # Standard projections
        init_custom_weight(self.query_w, is_residual_output=False)
        init_custom_weight(self.key_w,   is_residual_output=False)
        init_custom_weight(self.value_w, is_residual_output=False)

        # Residual output projection (scaled by depth)
        init_custom_weight(self.out_w, is_residual_output=True, num_layers=self.num_layers)



    def forward(
        self,
        x: torch.tensor,
        use_cache: bool = False,
        kv_cache: tuple[torch.Tensor, torch.Tensor] | None = None, 
    )-> tuple[torch.Tensor, tuple[torch.Tensor, torch.Tensor] | None]:
        
        batch_size, seq_len, embed_dim = x.shape 


        # -------------------------------------------------
        # 1. Linear projections (via einops.einsum)
        #    (batch_size, seq_len, embed_dim) @ W
        # →  (batch_size, seq_len, num_attn_heads * head_dim)
        #
        # nn.Linear.weight has shape (out_features, in_features), 
        # so the einsum contracts on embed_dim.
        # -------------------------------------------------
        q_vec = einsum(
            x, self.query_w.weight,
            "batch_size seq_len embed_dim, "
            "embed_dim num_attn_heads_times_head_dim"
            " -> "
            "batch_size seq_len num_attn_heads_times_head_dim",
        )

        k_vec = einsum(
            x, self.key_w.weight,
            "batch_size seq_len embed_dim, "
            "embed_dim num_attn_heads_times_head_dim"
            " -> "
            "batch_size seq_len num_attn_heads_times_head_dim",
        )

        v_vec = einsum(
            x, self.value_w.weight,
            "batch_size seq_len embed_dim, "
            "embed_dim num_attn_heads_times_head_dim"
            " -> "
            "batch_size seq_len num_attn_heads_times_head_dim",
        )


        # -------------------------------------------------
        # 2. Rearrange into multi-head form
        #    (batch_size, seq_len, num_attn_heads * head_dim)
        # →  (batch_size, num_attn_heads, seq_len, head_dim)
        # -------------------------------------------------
        q = rearrange(
            q_vec,
            "batch_size seq_len (num_attn_heads head_dim)"
            " -> "
            "batch_size num_attn_heads seq_len head_dim",
            num_attn_heads=self.num_attn_heads,
        )

        k = rearrange(
            k_vec,
            "batch_size seq_len (num_attn_heads head_dim)"
            " -> "
            "batch_size num_attn_heads seq_len head_dim",
            num_attn_heads=self.num_attn_heads,
        )

        v = rearrange(
            v_vec,
            "batch_size seq_len (num_attn_heads head_dim)"
            " -> "
            "batch_size num_attn_heads seq_len head_dim",
            num_attn_heads=self.num_attn_heads,
        )


        # if kv_cache is not None -> x:[B, 1, D] decode
        # if kv_cache is     None -> x:[B, T, D] prefill
        if use_cache:
            if kv_cache is not None:
                prev_k, prev_v = kv_cache
                # Concatenate on the sequence dimension
                k = torch.cat([prev_k, k], dim=2) # # (B, H, prev+cur, D)
                v = torch.cat([prev_v, v], dim=2)
            new_kv_cache = (k, v)
        else:
            new_kv_cache = None


        atten_scores = ...

        atten_scores_masked = ...

        softmaxed_atten_weights = ...

        atten_weighted_val = ...

        atten_out = ...

        return atten_out, new_kv_cache
        


In [ ]:
    def __init__(self, config):
        super().__init__()

        self.embed_dim = config["embed_dim"]
        self.num_attn_heads = config["num_attn_heads"]
        self.head_dim = self.embed_dim // self.num_attn_heads
        self.num_layers = config["num_layers"]          # needed for residual scaling

        assert self.embed_dim % self.num_attn_heads == 0, \
            "embed_dim must be divisible by num_attn_heads"

        # Weight matrices stored as (in_features, out_features)
        #   query/key/value : (embed_dim, num_heads * head_dim)
        #   out             : (num_heads * head_dim, embed_dim)
        self.query_w = nn.Parameter(torch.empty(self.embed_dim, self.num_attn_heads * self.head_dim))
        self.key_w   = nn.Parameter(torch.empty(self.embed_dim, self.num_attn_heads * self.head_dim))
        self.value_w = nn.Parameter(torch.empty(self.embed_dim, self.num_attn_heads * self.head_dim))
        self.out_w   = nn.Parameter(torch.empty(self.num_attn_heads * self.head_dim, self.embed_dim))

        self.reset_parameters()

        # Causal mask (True = positions that must be masked)
        mask = torch.triu(
            torch.ones(config["max_seq_len"], config["max_seq_len"], dtype=torch.bool),
            diagonal=1,
        )
        self.register_buffer("mask", mask, persistent=False)

    def reset_parameters(self):
        # Standard projections
        init_custom_weight(self.query_w, is_residual_output=False)
        init_custom_weight(self.key_w,   is_residual_output=False)
        init_custom_weight(self.value_w, is_residual_output=False)

        # Residual output projection (scaled by depth)
        init_custom_weight(self.out_w, is_residual_output=True, num_layers=self.num_layers)

    def forward(
        self,
        x: torch.Tensor,
        use_cache: bool = False,
        kv_cache: tuple[torch.Tensor, torch.Tensor] | None = None,
    ) -> tuple[torch.Tensor, tuple[torch.Tensor, torch.Tensor] | None]:
        """
        Args:
            x:          (batch_size, seq_len, embed_dim)
                        – prefill: seq_len = T
                        – decode : seq_len = 1
            use_cache:  whether to return / use KV cache
            kv_cache:   optional (prev_k, prev_v) each of shape
                        (batch_size, num_heads, prev_len, head_dim)

        Returns:
            atten_out:     (batch_size, seq_len, embed_dim)
            new_kv_cache:  (k, v) or None
        """
        batch_size, seq_len, _ = x.shape

        # ------------------------------------------------------------------
        # 1. Linear projections (einops.einsum)
        #    (batch, seq, embed_dim) @ W
        # →  (batch, seq, num_heads * head_dim)
        # ------------------------------------------------------------------
        q_vec = einsum(
            x, self.query_w,
            "batch_size seq_len embed_dim, "
            "embed_dim num_attn_heads_times_head_dim -> "
            "batch_size seq_len num_attn_heads_times_head_dim",
        )
        k_vec = einsum(
            x, self.key_w,
            "batch_size seq_len embed_dim, "
            "embed_dim num_attn_heads_times_head_dim -> "
            "batch_size seq_len num_attn_heads_times_head_dim",
        )
        v_vec = einsum(
            x, self.value_w,
            "batch_size seq_len embed_dim, "
            "embed_dim num_attn_heads_times_head_dim -> "
            "batch_size seq_len num_attn_heads_times_head_dim",
        )

        # ------------------------------------------------------------------
        # 2. Rearrange into multi-head form
        #    (batch, seq, num_heads * head_dim)
        # →  (batch, num_heads, seq, head_dim)
        # ------------------------------------------------------------------
        q = rearrange(
            q_vec,
            "batch_size seq_len (num_attn_heads head_dim) -> "
            "batch_size num_attn_heads seq_len head_dim",
            num_attn_heads=self.num_attn_heads,
        )
        k = rearrange(
            k_vec,
            "batch_size seq_len (num_attn_heads head_dim) -> "
            "batch_size num_attn_heads seq_len head_dim",
            num_attn_heads=self.num_attn_heads,
        )
        v = rearrange(
            v_vec,
            "batch_size seq_len (num_attn_heads head_dim) -> "
            "batch_size num_attn_heads seq_len head_dim",
            num_attn_heads=self.num_attn_heads,
        )

        # ------------------------------------------------------------------
        # 3. KV-cache handling
        # ------------------------------------------------------------------
        if use_cache:
            if kv_cache is not None:
                prev_k, prev_v = kv_cache
                # Concatenate on the sequence dimension
                k = torch.cat([prev_k, k], dim=2)   # (B, H, prev+cur, D)
                v = torch.cat([prev_v, v], dim=2)
            new_kv_cache = (k, v)
        else:
            new_kv_cache = None

        # Current key/value length (after possible concatenation)
        key_len = k.shape[2]

        # ------------------------------------------------------------------
        # 4. Scaled dot-product attention
        # ------------------------------------------------------------------
        # scores: (batch, num_heads, query_len, key_len)
        atten_scores = einsum(
            q, k,
            "batch_size num_attn_heads query_len head_dim, "
            "batch_size num_attn_heads key_len head_dim -> "
            "batch_size num_attn_heads query_len key_len",
        ) / math.sqrt(self.head_dim)

        # Causal mask
        # self.mask is True on the upper triangle (positions that must be blocked)
        if key_len <= self.mask.shape[0]:
            # Slice the pre-computed mask to the current key length
            # For decode (query_len=1) the relevant row is the last one
            causal_mask = self.mask[:key_len, :key_len]          # (key_len, key_len)
            # Broadcast to (1, 1, query_len, key_len) – we only need the last query_len rows
            causal_mask = causal_mask[-seq_len:, :]              # (query_len, key_len)
            atten_scores = atten_scores.masked_fill(causal_mask, float("-inf"))
        else:
            # Fallback if somehow longer than max_seq_len (should not happen)
            raise ValueError(f"Sequence length {key_len} exceeds max_seq_len")

        # Softmax over the key dimension
        softmaxed_atten_weights = torch.softmax(atten_scores, dim=-1)

        # Weighted sum of values → (batch, num_heads, query_len, head_dim)
        atten_weighted_val = einsum(
            softmaxed_atten_weights, v,
            "batch_size num_attn_heads query_len key_len, "
            "batch_size num_attn_heads key_len head_dim -> "
            "batch_size num_attn_heads query_len head_dim",
        )

        # ------------------------------------------------------------------
        # 5. Merge heads + output projection
        # ------------------------------------------------------------------
        # (batch, num_heads, seq, head_dim) → (batch, seq, num_heads * head_dim)
        concat_heads = rearrange(
            atten_weighted_val,
            "batch_size num_attn_heads seq_len head_dim -> "
            "batch_size seq_len (num_attn_heads head_dim)",
        )

        # Final linear: (batch, seq, num_heads*head_dim) @ out_w
        # → (batch, seq, embed_dim)
        atten_out = einsum(
            concat_heads, self.out_w,
            "batch_size seq_len num_attn_heads_times_head_dim, "
            "num_attn_heads_times_head_dim embed_dim -> "
            "batch_size seq_len embed_dim",
        )

        return atten_out, new_kv_cache

In [ ]:
model = Level3SingleHeadCausalAttention_Model()
configs = MODEL_CONFIGS

training_engine.train_and_save_model(model, configs, device, 100)
training_engine.plot_loss_history(configs)


In [ ]:
prompt = "there was a"
inference_engine.advanced_inference(model, configs, 20, prompt, device, eos_token_id=83)